|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Continuous batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: write the iteration-level scheduler<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(1)

Write the scheduler.

One function, one loop, and a hook for the admission policy so you can change
your mind about it in Exercise 4. This is stage 05 of the ladder with the GPU
replaced by a counter, which is the right way to get a scheduler correct
before you make it fast.

In [ ]:
### run this cell

N = 3000
B = 64
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=N).astype(int) + 1
capacity = B / lengths.mean()
arrive   = np.cumsum(rng.exponential(1/(0.7*capacity), size=N))

print(f'{N} requests, mean {lengths.mean():.0f} tokens, {B} slots')

# Exercise 1: the step loop

Four things happen, in this order, every step:

1. arrivals join the waiting queue
2. free slots are filled from it
3. every running sequence emits one token
4. anything that just finished leaves its slot immediately

Step 4 is the whole idea. A static batcher does it at the end of the batch.

In [ ]:
def run(arrive, lengths, B, pick):
  """pick(waiting) -> index into `waiting` of the request to admit next."""
  done, started = np.zeros(len(lengths)), np.zeros(len(lengths))
  t, nxt, running, waiting = 0.0, 0, {}, []
  busy = []

  while nxt < len(lengths) or waiting or running:
    while nxt < len(lengths) and arrive[nxt] <= t:
      waiting.append(nxt); nxt += 1

    while len(running) < B and waiting:
      r = waiting.pop(pick(waiting, lengths))
      running[r] = lengths[r]; started[r] = t

    if not running:
      t = arrive[nxt]; continue

    busy.append(len(running))
    t += 1.0
    for r in list(running):
      running[r] -= 1
      if running[r] == 0:
        done[r] = t; del running[r]

  return done, started, np.array(busy)

fcfs = lambda waiting, lengths: 0
done, started, busy = run(arrive, lengths, B, fcfs)
print(f'finished at step {done.max():,.0f}')

# Exercise 2: prove it is not lying

A scheduler that drops requests or hands out extra tokens will still produce
a plausible-looking throughput number. Check the invariants.

In [ ]:
assert (done > 0).all(), 'some request never finished'
assert np.allclose(done - started, lengths), 'wrong number of tokens somewhere'
assert (started >= arrive - 1e-9).all(), 'a request started before it arrived'
assert busy.max() <= B, 'more sequences running than there are slots'
print('all checks passed')
print(f'mean occupancy {busy.mean():.1f} of {B} slots ({100*busy.mean()/B:.0f}%)')

# Exercise 3: against static batching

In [ ]:
def static_batching(arrive, lengths, B):
  out = np.zeros(len(lengths)); t = 0.0; i = 0
  while i < len(lengths):
    b = np.arange(i, min(i+B, len(lengths)))
    t = max(t, arrive[b[-1]]); s = lengths[b].max()
    out[b] = t + s; t += s; i += B
  return out

ds = static_batching(arrive, lengths, B)
lat_s, lat_c = ds - arrive, done - arrive

print(f"{'':<12} {'makespan':>10} {'p50 lat':>9} {'p99 lat':>9}")
print(f"{'static':<12} {ds.max():>10,.0f} {np.median(lat_s):>9,.0f} {np.percentile(lat_s,99):>9,.0f}")
print(f"{'continuous':<12} {done.max():>10,.0f} {np.median(lat_c):>9,.0f} {np.percentile(lat_c,99):>9,.0f}")
print(f'\nthroughput {ds.max()/done.max():.2f}x, p99 latency {np.percentile(lat_s,99)/np.percentile(lat_c,99):.0f}x better')

# Exercise 4: change the admission policy

First-come-first-served is one choice. Try admitting the shortest request
instead, and look at what it does to each part of the distribution rather
than at the average.

One thing first: at 70% load your waiting queue is empty almost every step,
so `pick` never gets a choice and every policy scores the same. Overload the
server to 120% and the question becomes real. A scheduler is only a scheduler
when there is something to schedule.

In [ ]:
# a policy only matters when there is a queue. At 70% load there
# almost never is one, so overload the server first.
busy_arrive = np.cumsum(rng.exponential(1/(1.2*capacity), size=N))
base, _, _  = run(busy_arrive, lengths, B, fcfs)

# admit the request with the fewest tokens left to generate
sjf = lambda waiting, lengths: int(np.argmin(lengths[waiting]))

d2, _, _ = run(busy_arrive, lengths, B, sjf)

print(f"{'policy':<22} {'makespan':>10} {'p50':>8} {'p99':>9} {'worst':>9}")
for nm, d in (('first come first served', base), ('shortest job first', d2)):
  L = d - busy_arrive
  print(f'{nm:<22} {d.max():>10,.0f} {np.median(L):>8,.0f} {np.percentile(L,99):>9,.0f} {L.max():>9,.0f}')

### Three results

**Continuous batching wins on both axes at once.** That is rare. Usually
throughput and latency trade against each other, and here one policy
change improves both by large factors, because the old policy was
wasting rather than trading.

**Shortest-job-first cuts the median and ruins the tail.** It is the
classic scheduling result and it shows up here exactly as the textbooks
say: short requests overtake long ones, so most users are served sooner
and the longest request is starved. Look at the `worst` column.

**And shortest-job-first is not implementable anyway.** It asked
`lengths[waiting]`, which is the number of tokens the model has not
generated yet. A real server does not know that. It finds out when the
model emits a stop token.

That is worth sitting with, because it is the shape of most scheduling
in this field: the optimal policy needs the future, so you build
something that approximates it from what you can see. Stage 10 is where
you face that honestly, under a memory budget that can run out
mid-sequence.

    ./vc guide 5